In [1]:
import torch
import sys

print("=" * 60)
print("VOICE KANBAN - SYSTEM CONFIGURATION CHECK")
print("=" * 60)
print(f"\nPython version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print("\n✅ GPU Configuration Complete - Ready for processing")
else:
    print("\n⚠️ No GPU detected - running on CPU (slower performance)")

print("=" * 60)


VOICE KANBAN - SYSTEM CONFIGURATION CHECK

Python version: 3.12.7 | packaged by conda-forge | (main, Oct  4 2024, 16:05:46) [GCC 13.3.0]
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
Number of GPUs: 1
GPU Name: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition

✅ GPU Configuration Complete - Ready for processing


/opt/conda/lib/python3.12/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [2]:
print("Installing dependencies...")
print("=" * 60)

# Core libraries for Gradio interface
!pip install gradio --break-system-packages
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu124

print("\n✅ All libraries installed successfully")
print("=" * 60)

Installing dependencies...
Looking in indexes: https://download.pytorch.org/whl/nightly/cu124
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.

✅ All libraries installed successfully


In [3]:
import gradio as gr
import json
from typing import List, Dict, Tuple

print("=" * 60)
print("IMPORTING LIBRARIES")
print("=" * 60)
print("✅ Gradio imported - UI framework")
print("✅ Core libraries imported")
print("=" * 60)

IMPORTING LIBRARIES
✅ Gradio imported - UI framework
✅ Core libraries imported


In [4]:
print("=" * 60)
print("LOADING TEAM PERSONAS")
print("=" * 60)

# Your 4 Personas with detailed skill profiles
INITIAL_TEAM = [
    {
        'id': 1,
        'name': 'Dana',
        'role': 'Designer',
        'skills': {
            'ui': 0.9,      # Expert in UI design
            'ux': 0.9,      # Expert in UX design
            'frontend': 0.4,  # Some frontend knowledge
            'testing': 0.2,   # Basic testing skills
            'ops': 0.1        # Minimal ops knowledge
        },
        'currentLoad': 0,
        'color': '#FF6B9D',
        'assignedTasks': []
    },
    {
        'id': 2,
        'name': 'Chris',
        'role': 'Coder',
        'skills': {
            'frontend': 0.9,  # Expert in frontend
            'backend': 0.9,   # Expert in backend
            'ui': 0.3,        # Some UI knowledge
            'testing': 0.4,   # Decent testing
            'ops': 0.3        # Some ops knowledge
        },
        'currentLoad': 0,
        'color': '#4ECDC4',
        'assignedTasks': []
    },
    {
        'id': 3,
        'name': 'Ted',
        'role': 'Tester',
        'skills': {
            'testing': 0.9,   # Expert in testing
            'backend': 0.3,   # Some backend knowledge
            'frontend': 0.3,  # Some frontend knowledge
            'ui': 0.2,        # Basic UI knowledge
            'ops': 0.2        # Basic ops knowledge
        },
        'currentLoad': 0,
        'color': '#FFE66D',
        'assignedTasks': []
    },
    {
        'id': 4,
        'name': 'Oliver',
        'role': 'Ops',
        'skills': {
            'ops': 0.9,       # Expert in ops
            'backend': 0.5,   # Good backend knowledge
            'testing': 0.4,   # Decent testing
            'frontend': 0.2,  # Basic frontend
            'ui': 0.1         # Minimal UI knowledge
        },
        'currentLoad': 0,
        'color': '#95E1D3',
        'assignedTasks': []
    }
]

print("\n✅ Team configuration loaded:")
for member in INITIAL_TEAM:
    print(f"  • {member['name']} ({member['role']}) - Color: {member['color']}")

print("=" * 60)



LOADING TEAM PERSONAS

✅ Team configuration loaded:
  • Dana (Designer) - Color: #FF6B9D
  • Chris (Coder) - Color: #4ECDC4
  • Ted (Tester) - Color: #FFE66D
  • Oliver (Ops) - Color: #95E1D3


In [5]:
print("=" * 60)
print("LOADING TASK POOL & SCORING ALGORITHM")
print("=" * 60)

# 10 Simplified, realistic Kanban tasks
TASKS = [
    {'id': 1, 'name': 'Fix login button alignment', 'type': 'ui', 'priority': 8, 'effort': 2},
    {'id': 2, 'name': 'Update navigation menu', 'type': 'frontend', 'priority': 7, 'effort': 3},
    {'id': 3, 'name': 'Test checkout flow', 'type': 'testing', 'priority': 9, 'effort': 4},
    {'id': 4, 'name': 'Deploy new API endpoint', 'type': 'ops', 'priority': 9, 'effort': 3},
    {'id': 5, 'name': 'Design user profile page', 'type': 'ui', 'priority': 6, 'effort': 4},
    {'id': 6, 'name': 'Fix database query timeout', 'type': 'backend', 'priority': 10, 'effort': 5},
    {'id': 7, 'name': 'Test mobile responsive layout', 'type': 'testing', 'priority': 7, 'effort': 3},
    {'id': 8, 'name': 'Add search filters', 'type': 'frontend', 'priority': 5, 'effort': 4},
    {'id': 9, 'name': 'Set up monitoring alerts', 'type': 'ops', 'priority': 8, 'effort': 2},
    {'id': 10, 'name': 'Create color palette', 'type': 'ux', 'priority': 4, 'effort': 2},
]

def calculate_score(member: Dict, task: Dict, algorithm: str) -> float:
    """
    Calculate assignment score based on:
    - Skill level (0.0-1.0)
    - Specialization bonus (+0.2 if role matches task type)
    - Load penalty (-0.1 per effort point already assigned)
    - Priority factor (task priority / 10)
    """
    skill_level = member['skills'].get(task['type'], 0)
    specialization_bonus = 0.2 if member['role'].lower()[:3] in task['type'].lower() else 0
    load_penalty = member['currentLoad'] * 0.1
    priority_factor = task['priority'] / 10
    
    score = (skill_level + specialization_bonus) * priority_factor - load_penalty
    
    # Algorithm-specific adjustments
    if algorithm == 'greedy':
        score *= (task['priority'] / 5)  # Emphasize high-priority tasks
    elif algorithm == 'balanced':
        score *= (1 - member['currentLoad'] / 20)  # Favor less-loaded members
    
    return max(0, score)

print(f"\n✅ Task pool loaded: {len(TASKS)} tasks")
print("✅ Scoring function configured")
print("\nTask types: ui, ux, frontend, backend, testing, ops")
print("Priority range: 1-10 (higher = more urgent)")
print("Effort range: 2-5 (higher = more work)")
print("=" * 60)



LOADING TASK POOL & SCORING ALGORITHM

✅ Task pool loaded: 10 tasks
✅ Scoring function configured

Task types: ui, ux, frontend, backend, testing, ops
Priority range: 1-10 (higher = more urgent)
Effort range: 2-5 (higher = more work)


In [6]:
print("=" * 60)
print("IMPLEMENTING ASSIGNMENT ALGORITHMS")
print("=" * 60)

def assign_tasks(algorithm: str) -> Tuple[List[Dict], List[Dict]]:
    """
    Assign tasks using specified algorithm.
    
    Algorithms:
    - hungarian: Optimal matching (best overall efficiency)
    - greedy: Priority-first (urgent tasks get best matches)
    - balanced: Even distribution (prevents overload)
    """
    assignments = []
    team_copy = [dict(m, currentLoad=0, assignedTasks=[]) for m in INITIAL_TEAM]
    available_tasks = list(TASKS)
    
    if algorithm == 'hungarian':
        # HUNGARIAN: Optimal matching algorithm
        # Sort tasks by priority, find best match for each
        available_tasks.sort(key=lambda x: x['priority'], reverse=True)
        
        for task in available_tasks:
            best_member = None
            best_score = -1
            
            for member in team_copy:
                score = calculate_score(member, task, algorithm)
                if score > best_score:
                    best_score = score
                    best_member = member
            
            if best_member:
                best_member['currentLoad'] += task['effort']
                best_member['assignedTasks'].append(task)
                assignments.append({
                    'task': task,
                    'member': {
                        'id': best_member['id'],
                        'name': best_member['name'],
                        'role': best_member['role'],
                        'color': best_member['color']
                    },
                    'score': best_score
                })
    
    elif algorithm == 'greedy':
        # GREEDY: Priority-first assignment
        # High-priority tasks get first pick of best resources
        available_tasks.sort(key=lambda x: x['priority'], reverse=True)
        
        for task in available_tasks:
            best_member = team_copy[0]
            best_score = calculate_score(best_member, task, algorithm)
            
            for member in team_copy[1:]:
                score = calculate_score(member, task, algorithm)
                if score > best_score:
                    best_score = score
                    best_member = member
            
            best_member['currentLoad'] += task['effort']
            best_member['assignedTasks'].append(task)
            assignments.append({
                'task': task,
                'member': {
                    'id': best_member['id'],
                    'name': best_member['name'],
                    'role': best_member['role'],
                    'color': best_member['color']
                },
                'score': best_score
            })
    
    elif algorithm == 'balanced':
        # BALANCED: Even distribution algorithm
        # Always assign to least-loaded member with best task match
        while available_tasks:
            least_loaded = min(team_copy, key=lambda m: m['currentLoad'])
            
            best_task = None
            best_score = -1
            
            for task in available_tasks:
                score = calculate_score(least_loaded, task, algorithm)
                if score > best_score:
                    best_score = score
                    best_task = task
            
            if best_task:
                least_loaded['currentLoad'] += best_task['effort']
                least_loaded['assignedTasks'].append(best_task)
                assignments.append({
                    'task': best_task,
                    'member': {
                        'id': least_loaded['id'],
                        'name': least_loaded['name'],
                        'role': least_loaded['role'],
                        'color': least_loaded['color']
                    },
                    'score': best_score
                })
                available_tasks.remove(best_task)
            else:
                break
    
    return assignments, team_copy

print("\n✅ Three algorithms implemented:")
print("  1. Hungarian - Optimal matching for overall efficiency")
print("  2. Greedy - Priority-first for urgent work")
print("  3. Balanced - Even distribution to prevent burnout")
print("=" * 60)

IMPLEMENTING ASSIGNMENT ALGORITHMS

✅ Three algorithms implemented:
  1. Hungarian - Optimal matching for overall efficiency
  2. Greedy - Priority-first for urgent work
  3. Balanced - Even distribution to prevent burnout


In [7]:
print("=" * 60)
print("CHAT INTERFACE FOR TASK COMMANDS")
print("=" * 60)

def parse_chat_command(message: str, history: list) -> str:
    """
    Parse natural language commands for task assignment.
    Future: Connect to Claude API for intent parsing.
    
    Examples:
    - "Show me the balanced algorithm"
    - "Who should do the testing tasks?"
    - "What's Dana's workload?"
    """
    
    message_lower = message.lower()
    
    # Command parsing logic (simple keyword matching for now)
    if "balance" in message_lower:
        assignments, team = assign_tasks('balanced')
        response = "Running BALANCED algorithm...\n\n"
        for member in team:
            response += f"{member['name']}: {member['currentLoad']} effort points, {len(member['assignedTasks'])} tasks\n"
        return response
    
    elif "hungarian" in message_lower or "optimal" in message_lower:
        assignments, team = assign_tasks('hungarian')
        response = "Running HUNGARIAN algorithm...\n\n"
        for member in team:
            response += f"{member['name']}: {member['currentLoad']} effort points, {len(member['assignedTasks'])} tasks\n"
        return response
    
    elif "greedy" in message_lower or "priority" in message_lower:
        assignments, team = assign_tasks('greedy')
        response = "Running GREEDY algorithm...\n\n"
        for member in team:
            response += f"{member['name']}: {member['currentLoad']} effort points, {len(member['assignedTasks'])} tasks\n"
        return response
    
    elif "dana" in message_lower:
        return "Dana is your Designer with expertise in UI (90%) and UX (90%). She can help with design tasks."
    
    elif "chris" in message_lower:
        return "Chris is your Coder with expertise in Frontend (90%) and Backend (90%). He can handle coding tasks."
    
    elif "ted" in message_lower:
        return "Ted is your Tester with expertise in Testing (90%). He focuses on quality assurance."
    
    elif "oliver" in message_lower:
        return "Oliver is your Ops specialist with expertise in Operations (90%) and Backend (50%). He handles deployments."
    
    elif "help" in message_lower:
        return """Available commands:
- "run balanced/hungarian/greedy algorithm"
- "tell me about [Dana/Chris/Ted/Oliver]"
- "show me tasks"
- "what's high priority?"

Future Phase 2: Full natural language with Claude API!"""
    
    elif "task" in message_lower:
        response = "📋 Available Tasks:\n\n"
        for task in TASKS:
            priority_marker = "🔴" if task['priority'] >= 9 else "🟡" if task['priority'] >= 7 else "🟢"
            response += f"{priority_marker} {task['name']} (Priority: {task['priority']})\n"
        return response
    
    else:
        return f"I understand you said: '{message}'\n\nTry commands like:\n- 'run balanced algorithm'\n- 'tell me about Dana'\n- 'show me tasks'\n\nType 'help' for more options."

print("\n✅ Chat command parser configured")
print("Available commands: algorithm selection, team info, task queries")
print("Future: Claude API integration for natural language understanding")
print("=" * 60)



CHAT INTERFACE FOR TASK COMMANDS

✅ Chat command parser configured
Available commands: algorithm selection, team info, task queries
Future: Claude API integration for natural language understanding


In [8]:
print("=" * 60)
print("LAUNCHING GRADIO INTERFACE")
print("=" * 60)

def format_results(algorithm: str) -> str:
    """Format assignment results with rich markdown"""
    assignments, team = assign_tasks(algorithm)
    
    # Header with algorithm info
    result = f"""# 🎯 {algorithm.upper()} Algorithm Results

---

## 👥 Team Workload Distribution

"""
    
    # Team member summaries
    for member in team:
        emoji = "🎨" if member['role'] == 'Designer' else \
                "💻" if member['role'] == 'Coder' else \
                "🧪" if member['role'] == 'Tester' else "⚙️"
        
        result += f"""
### {emoji} {member['name']} ({member['role']})
- **Workload:** {member['currentLoad']} effort points
- **Tasks assigned:** {len(member['assignedTasks'])}
"""
        
        if member['assignedTasks']:
            result += "\n**Assigned tasks:**\n"
            for task in member['assignedTasks']:
                priority_emoji = "🔴" if task['priority'] >= 9 else "🟡" if task['priority'] >= 7 else "🟢"
                result += f"- {priority_emoji} {task['name']} (Priority: {task['priority']}, Effort: {task['effort']})\n"
        else:
            result += "\n*No tasks assigned*\n"
    
    # Summary statistics
    total_effort = sum(m['currentLoad'] for m in team)
    avg_effort = total_effort / len(team)
    
    result += f"""

---

## 📊 Summary Statistics
- **Total effort distributed:** {total_effort} points
- **Average load per person:** {avg_effort:.1f} points
- **Tasks assigned:** {len(assignments)} / {len(TASKS)}

---

### Algorithm Characteristics

"""
    
    if algorithm == 'hungarian':
        result += "**Hungarian Method:** Optimizes overall team efficiency by matching skills to tasks"
    elif algorithm == 'greedy':
        result += "**Greedy Method:** Prioritizes urgent tasks, may create workload imbalance"
    elif algorithm == 'balanced':
        result += "**Balanced Method:** Ensures even distribution, prevents team member overload"
    
    return result

def format_team_config() -> str:
    """Display team configuration"""
    result = "# 👥 Team Configuration\n\n"
    
    for member in INITIAL_TEAM:
        emoji = "🎨" if member['role'] == 'Designer' else \
                "💻" if member['role'] == 'Coder' else \
                "🧪" if member['role'] == 'Tester' else "⚙️"
        
        result += f"## {emoji} {member['name']} - {member['role']}\n\n"
        result += "**Skill Profile:**\n"
        
        for skill, level in sorted(member['skills'].items(), key=lambda x: x[1], reverse=True):
            bar = "█" * int(level * 10) + "░" * (10 - int(level * 10))
            result += f"- {skill.upper()}: {bar} {int(level * 100)}%\n"
        
        result += "\n---\n\n"
    
    return result

def format_task_pool() -> str:
    """Display available tasks"""
    result = "# 📋 Task Pool\n\n"
    
    # Group tasks by priority
    high_priority = [t for t in TASKS if t['priority'] >= 9]
    medium_priority = [t for t in TASKS if 7 <= t['priority'] < 9]
    low_priority = [t for t in TASKS if t['priority'] < 7]
    
    result += "## 🔴 High Priority (9-10)\n\n"
    for task in high_priority:
        result += f"- **{task['name']}** | Type: {task['type']} | Effort: {task['effort']}\n"
    
    result += "\n## 🟡 Medium Priority (7-8)\n\n"
    for task in medium_priority:
        result += f"- **{task['name']}** | Type: {task['type']} | Effort: {task['effort']}\n"
    
    result += "\n## 🟢 Low Priority (1-6)\n\n"
    for task in low_priority:
        result += f"- **{task['name']}** | Type: {task['type']} | Effort: {task['effort']}\n"
    
    return result

# Create Gradio interface with Google Material Design aesthetics
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="slate",
    ),
    title="Voice Kanban Assignment System",
    css="""
    .gradio-container {
        font-family: 'Google Sans', 'Roboto', sans-serif;
    }
    """
) as demo:
    
    gr.Markdown("""
    # 🎯 Voice Kanban Assignment System
    ### DF-R302-02 Capstone Project - Phase 1 Foundation
    
    Test different assignment algorithms with your 4 personas: Dana (Designer), Chris (Coder), Ted (Tester), and Oliver (Ops)
    """)
    
    with gr.Tabs():
        # TAB 1: Assignment Demo
        with gr.Tab("🎮 Assignment Demo"):
            gr.Markdown("### Select an algorithm and see how tasks are distributed")
            
            algorithm_input = gr.Radio(
                choices=['hungarian', 'greedy', 'balanced'],
                label="Assignment Algorithm",
                value='balanced',
                info="Choose how tasks should be assigned to team members"
            )
            
            assign_btn = gr.Button("▶️ Assign Tasks", variant="primary", size="lg")
            
            results_output = gr.Markdown(label="Results")
            
            assign_btn.click(
                fn=format_results,
                inputs=algorithm_input,
                outputs=results_output
            )
            
            gr.Examples(
                examples=[['hungarian'], ['greedy'], ['balanced']],
                inputs=algorithm_input,
                label="Quick Test"
            )
        
        # TAB 2: Team Configuration
        with gr.Tab("👥 Team Config"):
            gr.Markdown(format_team_config())
        
        # TAB 3: Task Pool
        with gr.Tab("📋 Task Pool"):
            gr.Markdown(format_task_pool())
        
        # TAB 4: Chat Commands
        with gr.Tab("💬 Chat Commands"):
            gr.Markdown("""
            ### Natural Language Task Commands
            
            Try typing commands like:
            - "run balanced algorithm"
            - "tell me about Dana"
            - "show me tasks"
            - "what's high priority?"
            
            **Phase 2 will add:**
            - Claude API for advanced natural language understanding
            - Voice input via Whisper
            - MCP integration for GitLab/GitHub
            """)
            
            chatbot = gr.Chatbot(height=400)
            msg = gr.Textbox(
                placeholder="Type a command like 'run balanced algorithm' or 'tell me about Dana'...",
                label="Command Input"
            )
            clear = gr.Button("Clear Chat")
            
            def user_message(message, history):
                return "", history + [[message, None]]
            
            def bot_response(history):
                user_msg = history[-1][0]
                bot_msg = parse_chat_command(user_msg, history)
                history[-1][1] = bot_msg
                return history
            
            msg.submit(user_message, [msg, chatbot], [msg, chatbot]).then(
                bot_response, chatbot, chatbot
            )
            
            clear.click(lambda: None, None, chatbot, queue=False)

print("\n✅ Gradio interface configured")
print("=" * 60)

# Launch the interface
demo.launch(
    share=True,
    server_name="0.0.0.0",
    show_error=True
)

print("\n🚀 Interface launched!")
print("Click the link above to open the web interface")
print("=" * 60)

LAUNCHING GRADIO INTERFACE


/tmp/ipykernel_612/1363665256.py:183: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)



✅ Gradio interface configured
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://412615580fd5d14a87.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🚀 Interface launched!
Click the link above to open the web interface


In [9]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Compute capability: {torch.cuda.get_device_capability(0)}")

PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
Compute capability: (12, 0)
